In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Colour palette used throughout
STATUS_ORDER  = ['Worsened', 'Stayed the same', 'Improved']
STATUS_COLORS = ['#C0392B', '#F39C12', '#27AE60']

plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False})
print("✅ Libraries loaded")


## 5. Modelling

In [ ]:
# MODEL 1: Decision Tree — simple, fully explainable
# max_depth=8 stops it from memorising the training data
# class_weight='balanced' compensates for the 52.6% Worsened majority
dt = DecisionTreeClassifier(max_depth=8, class_weight='balanced', random_state=42)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)
dt_f1 = f1_score(y_test, dt_pred, average='weighted')

print(f"🌳 Decision Tree — Weighted F1: {dt_f1:.4f}")
print(classification_report(y_test, dt_pred, target_names=target_le.classes_))


In [ ]:
# MODEL 2: Random Forest — 200 trees voting together for better accuracy
# This is our PRIMARY model
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1          # uses all CPU cores
)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_f1 = f1_score(y_test, rf_pred, average='weighted')

print(f"🌲 Random Forest — Weighted F1: {rf_f1:.4f}")
print(f"   Improvement over Decision Tree: +{rf_f1 - dt_f1:.4f}")
print()
print(classification_report(y_test, rf_pred, target_names=target_le.classes_))


## 6. Model Evaluation

In [ ]:
# Chart 6 — Confusion matrices for both models side by side
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, pred, title in zip(
    axes,
    [dt_pred, rf_pred],
    [f'Decision Tree  (Weighted F1 = {dt_f1:.3f})', f'Random Forest  (Weighted F1 = {rf_f1:.3f})']
):
    cm     = confusion_matrix(y_test, pred)
    cm_pct = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis] * 100
    sns.heatmap(cm_pct, annot=True, fmt='.1f', ax=ax,
                xticklabels=target_le.classes_, yticklabels=target_le.classes_,
                cmap='RdYlGn', vmin=0, vmax=100,
                annot_kws={'size': 11, 'weight': 'bold'})
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

fig.suptitle('Confusion Matrices — % of Actual Class Correctly Identified',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Per-class accuracy from Random Forest
rf_cm = confusion_matrix(y_test, rf_pred)
print("Random Forest — per class recall:")
for i, cls in enumerate(target_le.classes_):
    pct = rf_cm[i, i] / rf_cm[i].sum() * 100
    print(f"  {cls}: {pct:.1f}% correctly identified")


## 7. Feature Importance — Key Drivers

In [ ]:
# Chart 7 — What drives financial outcomes most?
feat_imp = pd.Series(rf.feature_importances_, index=X.columns) \
             .sort_values(ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(9, 7))
bar_colors = ['#7B1C2E' if i >= len(feat_imp) - 5 else '#BDC3C7'
              for i in range(len(feat_imp))]
ax.barh(feat_imp.index, feat_imp.values, color=bar_colors, edgecolor='white', height=0.6)
for i, (idx, val) in enumerate(feat_imp.items()):
    ax.text(val + 0.001, i, f'{val:.3f}', va='center', fontsize=9)
ax.set_title('Top 15 Features — Random Forest Feature Importance',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.legend(handles=[mpatches.Patch(color='#7B1C2E', label='Top 5 Key Drivers')], fontsize=9)
plt.tight_layout()
plt.show()

print("Top 5 drivers of financial status:")
for feat, score in feat_imp.tail(5)[::-1].items():
    print(f"  {feat}: {score:.4f}")


## 8. Key Findings & Recommendations

### What the model tells us

| # | Feature | What it means for policy |
|---|---------|--------------------------|
| 1 | **location_type / county** | Where you live determines your financial trajectory — geographic targeting is critical |
| 2 | **prodsum1** | Using more financial products builds resilience — promote formal financial inclusion |
| 3 | **Age** | Middle-aged adults are most vulnerable — peak expenses, variable income |
| 4 | **barriers_bank** | Banking access barriers directly predict worsened outcomes |
| 5 | **monthly_income** | Income is the root driver — low earners rarely recover without intervention |

### Recommendations

1. **Policymakers** — Income support programmes (cash transfers, minimum wage) have the highest impact
2. **County Governments** — Allocate resources by county-level financial deterioration data
3. **Banks & MFIs** — Expand affordable products to low-income earners; prodsum1 shows product use = resilience
4. **NGOs** — Build financial shock resilience through emergency funds and microinsurance


In [ ]:
# Final results summary
print("=" * 55)
print("   DATASPRINT 2026 — FINAL RESULTS")
print("=" * 55)
print(f"  Dataset : 20,871 Kenyan adults (2024 FinAccess)")
print(f"  Problem : Multiclass Classification (3 classes)")
print(f"  Split   : 80% train / 20% test (stratified)")
print()
print(f"  Decision Tree  Weighted F1 = {dt_f1:.4f}")
print(f"  Random Forest  Weighted F1 = {rf_f1:.4f}  ← PRIMARY MODEL")
print()
print(f"  Top drivers: location_type, county, prodsum1, Age, barriers_bank")
print("=" * 55)
print("  Learn. Build. Launch. — Strathmore Data Community")
print("=" * 55)
